# Model Visualization

Set `MODE` and `RUN_DIR` below, then run cells from top to bottom.  
Use `MODE = "aas_v2"` for AAS/SAM-Adapter models trained under this repo's `save/...`.  
Use `MODE = "official_sam2_adapter"` only for models trained under `third_party/SAM2-Adapter-PyTorch/save/...`.


In [ ]:
import os
import sys
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader

ROOT = Path('/home/una/aas')

# Choose one:
MODE = 'aas_v2'  # 'aas_v2' or 'official_sam2_adapter'

# Recent SAM2-Adapter examples:
RUN_DIR = ROOT / '/home/una/aas/save/v2_polyp_cont_S1_0519_train'
CKPT_NAME = 'model_epoch_best.pth'

# Official SAM2-Adapter examples:
# MODE = 'official_sam2_adapter'
# RUN_DIR = ROOT / 'third_party/SAM2-Adapter-PyTorch/save/sam2_adapter_camo'
# RUN_DIR = ROOT / 'third_party/SAM2-Adapter-PyTorch/save/sam2_adapter_polyp_0521'
# CKPT_NAME = 'model_epoch_best.pth'

NUM_SAMPLES = 50

THRESHOLD = 0.5
BATCH_SIZE = 1

# AAS/SAM-Adapter ablation: keep the checkpoint weights, but turn off adapter residuals.
# This visualizes the same trained SAM segmentation head without adapter contribution.
DISABLE_ADAPTER = False

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
print('run_dir:', RUN_DIR)
print('disable_adapter:', DISABLE_ADAPTER)


In [ ]:
def clear_project_modules():
    """Avoid mixing local AAS modules with official SAM2-Adapter modules in one kernel."""
    prefixes = ('models', 'datasets', 'utils', 'sod_metric')
    for name in list(sys.modules):
        if name in prefixes or any(name.startswith(prefix + '.') for prefix in prefixes):
            del sys.modules[name]


def setup_imports(mode):
    clear_project_modules()
    for p in [str(ROOT), str(ROOT / 'third_party/SAM2-Adapter-PyTorch')]:
        while p in sys.path:
            sys.path.remove(p)

    if mode == 'official_sam2_adapter':
        official_dir = ROOT / 'third_party/SAM2-Adapter-PyTorch'
        sys.path.insert(0, str(official_dir))
        os.chdir(official_dir)
    elif mode == 'aas_v2':
        sys.path.insert(0, str(ROOT))
        os.chdir(ROOT)
    else:
        raise ValueError(f'Unknown MODE: {mode}')

    import models
    import datasets

    if mode == 'aas_v2':
        import models.repeated_adapter_sam  # registers repeated_adapter_sam

    return models, datasets


models, datasets = setup_imports(MODE)
print('cwd:', os.getcwd())
print('models module:', models.__file__)
print('datasets module:', datasets.__file__)


In [ ]:
config_path = RUN_DIR / 'config.yaml'
ckpt_path = RUN_DIR / CKPT_NAME

with open(config_path, 'r') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

if 'test_dataset' not in config or config.get('test_dataset') is None:
    print('test_dataset not found; using val_dataset')
    config['test_dataset'] = config['val_dataset']

print('config:', config_path)
print('checkpoint:', ckpt_path)
print('model:', config['model']['name'])
print('eval_type:', config.get('eval_type'))


In [ ]:
def load_state_dict_flexible(model, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if isinstance(checkpoint, dict) and 'model' in checkpoint:
        checkpoint = checkpoint['model']
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        checkpoint = checkpoint['state_dict']
    checkpoint = {
        (key[7:] if key.startswith('module.') else key): value
        for key, value in checkpoint.items()
    }
    missing, unexpected = model.load_state_dict(checkpoint, strict=False)
    print(f'loaded checkpoint: missing={len(missing)}, unexpected={len(unexpected)}')
    if missing:
        print('missing examples:', missing[:8])
    if unexpected:
        print('unexpected examples:', unexpected[:8])


model = models.make(config['model']).to(DEVICE)
load_state_dict_flexible(model, ckpt_path, DEVICE)

if DISABLE_ADAPTER and hasattr(model, 'image_encoder') and hasattr(model.image_encoder, 'set_adapter_gamma'):
    depth = config['model']['args']['encoder_mode'].get('depth', 12)
    model.image_encoder.set_adapter_gamma([0.0] * depth)
    print(f'adapter residual disabled: gamma={[0.0] * depth}')

model.eval()

num_params = sum(p.numel() for p in model.parameters())
print(f'model params: {num_params:,}')


In [ ]:
def make_loader(spec):
    dataset = datasets.make(spec['dataset'])
    dataset = datasets.make(spec['wrapper'], args={'dataset': dataset})
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0), dataset

loader, dataset = make_loader(config['test_dataset'])
print('dataset size:', len(dataset))
first = dataset[0]
print('sample keys:', list(first.keys()))
for k, v in first.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape), v.dtype, float(v.min()), float(v.max()))


In [ ]:
def tensor_to_image(tensor):
    arr = tensor.detach().float().cpu()[0].numpy().transpose(1, 2, 0)
    arr = arr - arr.min()
    denom = arr.max() if arr.max() > 0 else 1.0
    arr = arr / denom
    return (arr * 255).clip(0, 255).astype(np.uint8)


def upsample_like(pred, gt):
    if pred.shape[-2:] != gt.shape[-2:]:
        pred = F.interpolate(pred, size=gt.shape[-2:], mode='bilinear', align_corners=False)
    return pred


def visualize_batch_sample(inp, gt, pred, threshold=0.5, title=None):
    pred = upsample_like(pred, gt)
    inp_img = tensor_to_image(inp)
    gt_mask = gt[0, 0].detach().float().cpu().numpy()
    pred_prob = pred[0, 0].detach().float().cpu().numpy()
    pred_bin = (pred_prob > threshold).astype(np.uint8)

    print('pred range:', float(pred_prob.min()), float(pred_prob.max()), 'mean:', float(pred_prob.mean()))
    print('positive pixels:', int(pred_bin.sum()), '/', pred_bin.size)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(inp_img)
    axes[0].set_title('Input')
    axes[0].axis('off')

    axes[1].imshow(gt_mask, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title('GT')
    axes[1].axis('off')

    im = axes[2].imshow(pred_prob, cmap='hot', vmin=0, vmax=1)
    axes[2].set_title('Prediction probability')
    axes[2].axis('off')
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    axes[3].imshow(pred_bin, cmap='gray', vmin=0, vmax=1)
    axes[3].set_title(f'Prediction > {threshold}')
    axes[3].axis('off')

    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
with torch.inference_mode():
    shown = 0
    for i, batch in enumerate(loader):
        if shown >= NUM_SAMPLES:
            break
        for k, v in batch.items():
            if torch.is_tensor(v):
                batch[k] = v.to(DEVICE)

        inp = batch['inp']
        inp = batch['inp']
        gt = batch['gt']
        pred = torch.sigmoid(model.infer(inp))

        print(f'\n=== sample {i + 1} ===')
        visualize_batch_sample(inp, gt, pred, threshold=THRESHOLD, title=f'{MODE}: sample {i + 1}')
        shown += 1

print(f'visualized {shown} samples')
